# FINS notebook 8 — quasi-geostrophic flow from pyqg: FNO2d emulators (Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp61, PG8, 09.10) before any of these runs. pyqg's barotropic model at 128²,
regimes f-plane and beta-plane:
* cell 2: for each regime and seed 42–44, pyqg generates 256 training trajectories, an FNO2d stepper is trained by
  exp45's recipe (modes 24, 40 epochs), then FINS's polynomial pencil runs through its tangent on pyqg's data.
  Results arrive stepper by stepper.
* cell 3 shows progress; cell 4 copies the outputs file by file to local disk (remounting Drive if it drops) and zips
  the json and logs to `fins_nb8.zip` (small: upload this one to the chat) and everything, with the steppers and
  pencils, to `fins_nb8_full.zip`, both in /content and on My Drive.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down). After a
disconnect rerun cells 1 and 2: finished runs are skipped, finished steppers are reused. Cell 1 installs neuraloperator
and pyqg (GitHub master) and stops if exp61 does not import; cell 2 ends by listing missing results (each failure prints its log's last lines).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp61), dependencies (neuraloperator, pyqg from GitHub)
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp61(z):
    with zipfile.ZipFile(z) as zz:
        return 'fins_repo/experiments/exp61_pyqg.py' in zz.namelist()


ZIP_PATH = newest('fins_repo_full*.zip', has_exp61)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp61: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'torchdiffeq'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/pyqg/pyqg.git'], check=False)
chk = subprocess.run([sys.executable, '-c', 'import exp61_pyqg'], cwd=EXPDIR, capture_output=True, text=True)
if chk.returncode:
    raise SystemExit('exp61 does not import here:\n' + chk.stderr[-2000:])
print('exp61 imports (pyqg found)')
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')
RUNS = os.path.join(R, 'pyqg_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)
REGIMES, SEEDS = ('f', 'beta'), (42, 43, 44)


def go(stem, cmd):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(RUNS, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if 'quotients' in l or 'FNO rollout' in l or 'scaling space' in l] if rc == 0 \
        else lines[-6:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:140] for k in keep[-6:]], sep='\n   ')

In [ ]:
# 2. per regime and seed: the FNO2d stepper (trained if missing), FINS through it (exp61's registered command)
for reg in REGIMES:
    for s in SEEDS:
        stem = os.path.join(RUNS, f'fins-fno_{reg}_q_s{s}')
        go(stem, ['exp61_pyqg.py', 'fins', '--op', 'fno', '--train-fno-if-missing', '--regime', reg, '--var', 'q',
                  '--data', 'pyqg', '--n', '128', '--every', '8', '--seed', str(s), '--save', stem])
left = [f'{reg} s {s}' for reg in REGIMES for s in SEEDS
        if not os.path.exists(os.path.join(RUNS, f'fins-fno_{reg}_q_s{s}.json'))]
print('all 6 results present' if not left else 'MISSING (see the logs above): ' + ', '.join(left))

In [ ]:
# 3. progress
for reg in REGIMES:
    for s in SEEDS:
        stem = os.path.join(RUNS, f'fins-fno_{reg}_q_s{s}')
        fno = os.path.join(R, 'pyqg', f'fno2d_{reg}_q_n128_s{s}.pt')
        log = os.path.join(RUNS, 'logs', f'fins-fno_{reg}_q_s{s}.log')
        ep = [l for l in open(log).read().splitlines() if 'epoch' in l] if os.path.exists(log) else []
        print(f'{reg} s {s}: result {os.path.exists(stem + ".json")}, stepper {os.path.exists(fno)}',
              '|', ep[-1].strip()[:80] if ep else '')

In [ ]:
# 4. copy the outputs file by file to local disk (remount if Drive drops); fins_nb8.zip (json, logs) and fins_nb8_full.zip
local = '/content/fins_nb8'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(os.path.join(RUNS, 'fins-fno_*')) + glob.glob(os.path.join(RUNS, 'logs', 'fins-fno_*'))
files += glob.glob(os.path.join(R, 'pyqg', 'fno2d_*.pt'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
small, out = '/content/fins_nb8.zip', '/content/fins_nb8_full.zip'
for path, keep in ((small, lambda f: f.endswith('.json') or f.endswith('.log')), (out, lambda f: True)):
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
            if os.path.isfile(f) and keep(f):
                z.write(f, os.path.relpath(f, local))
    print(path, os.path.getsize(path), 'bytes')
try:
    shutil.copy2(small, '/content/drive/MyDrive/fins_nb8.zip')
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb8_full.zip')
    print('copied to MyDrive: fins_nb8.zip (json and logs: upload this one to the chat) and fins_nb8_full.zip')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb8.zip from the file pane:', e)